# News Recommender — Login

Run the cells below in order.

- **Returning user?** Choose `[1] Log in`.
- **New here?** Choose `[2] Create a new account`.

After a successful login your session is saved automatically.  
Then open **FinalRecommender.ipynb** and run it to start reading.

In [ ]:
# Cell 1 — Setup

import hashlib
import json
import os
from IPython.display import clear_output

# Both files live one level up (in the News Recommender root folder,
# next to the Notebooks/ directory).
BASE_DIR    = os.path.abspath(os.path.join(os.getcwd(), ".."))
USERS_FILE  = os.path.join(BASE_DIR, "users.json")
SESSION_FILE = os.path.join(BASE_DIR, "current_session.json")

DIVIDER = "=" * 63
LINE    = "-" * 63

print(f"Users file   : {USERS_FILE}")
print(f"Session file : {SESSION_FILE}")
print("Setup complete.")

In [ ]:
# Cell 2 — Core auth functions

def _hash(password):
    return hashlib.sha256(password.encode("utf-8")).hexdigest()

def _load_users():
    if not os.path.exists(USERS_FILE):
        return {"users": []}
    with open(USERS_FILE, "r") as f:
        return json.load(f)

def _save_users(data):
    with open(USERS_FILE, "w") as f:
        json.dump(data, f, indent=2)

def _find_user(data, email):
    for u in data["users"]:
        if u["email"].lower() == email.lower():
            return u
    return None

def _save_session(user):
    """Write the logged-in user's email to current_session.json."""
    with open(SESSION_FILE, "w") as f:
        json.dump({"email": user["email"], "name": user["name"]}, f, indent=2)

def _clear_session():
    """Remove the session file (used on logout or failed login)."""
    if os.path.exists(SESSION_FILE):
        os.remove(SESSION_FILE)

print("Auth functions loaded.")

In [ ]:
# Cell 3 — Run login / register

_clear_session()   # always start fresh

clear_output(wait=True)
print("\n" + DIVIDER)
print("  NEWS RECOMMENDER  —  SIGN IN")
print(DIVIDER)
print("\n  [1]  Log in to an existing account")
print("  [2]  Create a new account\n")

while True:
    choice = input("  Enter 1 or 2: ").strip()
    if choice in ("1", "2"):
        break
    print("  Please enter 1 or 2.")

data = _load_users()

# ── REGISTER ────────────────────────────────────────────────────────────────
if choice == "2":
    print("\n" + LINE)
    print("  CREATE ACCOUNT\n")

    name = input("  Your name             : ").strip() or "User"

    while True:
        email = input("  Email address         : ").strip()
        if "@" in email and "." in email:
            break
        print("  Please enter a valid email address.")

    if _find_user(data, email):
        print("\n  An account with this email already exists.")
        print("  Switching to log in...\n")
        choice = "1"   # fall through to login block
    else:
        while True:
            password = input("  Password (min 6 chars) : ").strip()
            if len(password) >= 6:
                break
            print("  Password must be at least 6 characters.")

        new_user = {
            "name":          name,
            "email":         email,
            "password_hash": _hash(password),
            "history":       []
        }
        data["users"].append(new_user)
        _save_users(data)
        _save_session(new_user)

        clear_output(wait=True)
        print("\n" + DIVIDER)
        print(f"  Account created! Welcome, {name}.")
        print(f"  Session saved. Open FinalRecommender.ipynb to start reading.")
        print(DIVIDER + "\n")
        choice = None   # done

# ── LOGIN ────────────────────────────────────────────────────────────────────
if choice == "1":
    print("\n" + LINE)
    print("  LOG IN\n")

    logged_in = False
    for attempt in range(3):
        email    = input("  Email address : ").strip()
        password = input("  Password      : ").strip()

        user = _find_user(data, email)
        if user and user["password_hash"] == _hash(password):
            _save_session(user)
            history_count = len(user.get("history", []))

            clear_output(wait=True)
            print("\n" + DIVIDER)
            print(f"  Welcome back, {user['name']}!")
            if history_count:
                print(f"  {history_count} article(s) saved from your previous sessions.")
            print(f"  Session saved. Open FinalRecommender.ipynb to start reading.")
            print(DIVIDER + "\n")
            logged_in = True
            break

        remaining = 2 - attempt
        if remaining > 0:
            print(f"\n  Incorrect email or password. {remaining} attempt(s) remaining.\n")
        else:
            print("\n  Too many failed attempts. Please re-run this cell to try again.")